In [1]:
# =================================================================================================
# NHANES 2013-2014
# Sleep Disorders Questionnaire
#
# File:
#   SLQ_H.xpt
#
# Component:
#   Questionnaire - Sleep Disorders
#
# Purpose:
#
#   Convert NHANES SLQ_H XPT -> fidelity-preserving corrected CSV.
#
#
# ATTACHED SOURCE FINDINGS
# -------------------------------------------------------------------------------------------------
#
#   Rows                         = 6,464
#   Columns                      = 4
#
#   Variables:
#
#       SEQN
#       SLD010H
#       SLQ050
#       SLQ060
#
#   SEQN range                   = 73557-83729
#   Missing SEQN                 = 0
#   Duplicate SEQN               = 0
#
#   SAS XPORT tiny-zero artifact = 0
#   True numeric zeros           = 0
#
#   Genuine decimal variables    = NONE
#
#
# FIDELITY RULES
# -------------------------------------------------------------------------------------------------
#
#   1. Remove unnecessary ".0" from SEQN.
#
#   2. Remove unnecessary ".0" from other proven whole-number/code fields.
#
#   3. Do NOT replace anything with zero.
#
#   4. Do NOT round.
#
#   5. Do NOT use float_format=.
#
#   6. Preserve missing-value positions exactly.
#
#   7. Convert ONLY fields proven to contain whole-number values.
#
#   8. Validate against official CDC frequencies.
#
#   9. Validate XPT -> CSV numeric round-trip fidelity.
#
#
# GENERAL NHANES PIPELINE RULE
# -------------------------------------------------------------------------------------------------
#
#   Genuine decimal variables from Dietary, Laboratory, Examination,
#   Demographics, etc. must remain AS IS.
#
#   This particular SLQ_H file contains NO genuine decimal variables.
#
# =================================================================================================


# =================================================================================================
# 1. IMPORTS
# =================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path


# =================================================================================================
# 2. PATHS
# =================================================================================================

BASE_DIR = Path().resolve().parent


DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Questionnnair"
)


XPT_FILE = (
    DATA_DIR
    / "SLQ_H.xpt"
)


CSV_FILE = (
    DATA_DIR
    / "SLQ_H_corrected.csv"
)


# =================================================================================================
# 3. READ ORIGINAL XPT
# =================================================================================================

df = pd.read_sas(
    XPT_FILE,
    format="xport",
    encoding="latin1"
)


#
# Preserve untouched pandas-decoded source.
#
# We use this later to prove that no scientific/source value
# was accidentally changed during cleaning/export.
#

source_df = df.copy()


print("=" * 110)

print(
    "NHANES 2013-2014 SLQ_H XPT -> CSV"
)

print("=" * 110)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Column names:",
    df.columns.tolist()
)


# =================================================================================================
# 4. EXPECTED OFFICIAL STRUCTURE
# =================================================================================================

EXPECTED_COLUMNS = [

    "SEQN",

    "SLD010H",

    "SLQ050",

    "SLQ060"

]


EXPECTED_ROWS = 6464


if (
    df.columns.tolist()
    !=
    EXPECTED_COLUMNS
):

    raise ValueError(
        "SLQ_H column structure differs from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != EXPECTED_ROWS:

    raise ValueError(
        "Unexpected SLQ_H row count.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(df):,}"
    )


print(
    "PASS: Structure = 6,464 rows x 4 columns."
)


# =================================================================================================
# 5. VERIFY ALL FOUR FIELDS LOADED NUMERICALLY
# =================================================================================================

numeric_columns = (

    df.select_dtypes(
        include=[np.number]
    )

    .columns

    .tolist()

)


if (
    numeric_columns
    !=
    EXPECTED_COLUMNS
):

    raise ValueError(
        "Expected all four SLQ_H fields "
        "to load numerically from XPT."
    )


print(
    "PASS: All four fields loaded numerically."
)


# =================================================================================================
# 6. CHECK KNOWN SAS XPORT TINY-ZERO ARTIFACT
# =================================================================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


tiny_counts = (

    df[EXPECTED_COLUMNS]

    .eq(TINY_VALUE)

    .sum()

)


tiny_total = int(
    tiny_counts.sum()
)


print(
    "\n--- XPORT TINY-ZERO CHECK ---"
)


print(
    tiny_counts.to_string()
)


print(
    "\nTotal tiny XPORT values:",
    tiny_total
)


#
# IMPORTANT:
#
# The actual attached SLQ_H contains zero tiny artifacts.
#
# Therefore there is NO zero-restoration step for this file.
#

if tiny_total != 0:

    raise ValueError(
        "STOP: Unexpected tiny XPORT values were found.\n\n"
        "The validated attached SLQ_H contains zero tiny artifacts.\n"
        "Do not automatically convert these values to zero."
    )


print(
    "PASS: No SAS XPORT tiny-zero artifacts."
)


# =================================================================================================
# 7. TRUE NUMERIC ZERO CHECK
# =================================================================================================

zero_counts = (

    df[EXPECTED_COLUMNS]

    .eq(0)

    .sum()

)


zero_total = int(
    zero_counts.sum()
)


print(
    "\n--- TRUE NUMERIC ZERO CHECK ---"
)


print(
    zero_counts.to_string()
)


print(
    "\nTotal true numeric zeros:",
    zero_total
)


if zero_total != 0:

    raise ValueError(
        "Unexpected numeric zero observations found in SLQ_H."
    )


print(
    "PASS: No zero-coded observations exist in SLQ_H."
)


# =================================================================================================
# 8. FRACTIONAL VALUE DISCOVERY
# =================================================================================================

#
# np.round() is used ONLY for diagnostic comparison.
#
# It does NOT modify the dataframe.
#

print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)


fractional_counts = {}


for col in EXPECTED_COLUMNS:

    values = (

        df[col]

        .dropna()

        .to_numpy(
            dtype=float
        )

    )


    if len(values) == 0:

        fractional_count = 0


    else:

        fractional_count = int(

            (
                ~np.isclose(

                    values,

                    np.round(
                        values
                    ),

                    rtol=0,

                    atol=1e-12

                )
            )

            .sum()

        )


    fractional_counts[
        col
    ] = fractional_count


    print(
        f"{col}: "
        f"{fractional_count:,} fractional values"
    )


fractional_variables = {

    col: count

    for col, count in fractional_counts.items()

    if count > 0

}


if fractional_variables:

    raise ValueError(
        "Unexpected genuine fractional values found:\n"
        + str(
            fractional_variables
        )
    )


print(
    "PASS: No genuine decimal variables exist in SLQ_H."
)


# =================================================================================================
# 9. FIELD TYPE PLAN
# =================================================================================================

#
# All 4 fields have been proven mathematically integral.
#

INTEGER_COLUMNS = [

    "SEQN",

    "SLD010H",

    "SLQ050",

    "SLQ060"

]


DECIMAL_COLUMNS = []


print(
    "\n--- FIELD TYPE PLAN ---"
)


print(
    "Whole-number/code fields:",
    INTEGER_COLUMNS
)


print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


# =================================================================================================
# 10. SEQN SOURCE VALIDATION
# =================================================================================================

seqn_min = int(
    df["SEQN"]
    .min()
)


seqn_max = int(
    df["SEQN"]
    .max()
)


seqn_missing = int(

    df["SEQN"]

    .isna()

    .sum()

)


seqn_duplicate = int(

    df["SEQN"]

    .duplicated()

    .sum()

)


print(
    "\n--- SEQN SOURCE CHECK ---"
)


print(
    "Minimum:",
    seqn_min
)


print(
    "Maximum:",
    seqn_max
)


print(
    "Missing:",
    seqn_missing
)


print(
    "Duplicates:",
    seqn_duplicate
)


if seqn_min != 73557:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if seqn_max != 83729:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if seqn_missing != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if seqn_duplicate != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 73557-83729 validated."
)


# =================================================================================================
# 11. CDC FREQUENCY CHECK - SLD010H
# =================================================================================================

#
# CDC official 2013-2014 frequencies:
#
#   2-11 hours        = 6,408
#   12+ hours         =    46
#   Refused           =     0
#   Don't know        =     7
#   Missing           =     3
#
#   Total             = 6,464
#

sld010h_2_to_11 = int(

    df["SLD010H"]

    .between(
        2,
        11
    )

    .sum()

)


sld010h_12 = int(

    df["SLD010H"]

    .eq(12)

    .sum()

)


sld010h_77 = int(

    df["SLD010H"]

    .eq(77)

    .sum()

)


sld010h_99 = int(

    df["SLD010H"]

    .eq(99)

    .sum()

)


sld010h_missing = int(

    df["SLD010H"]

    .isna()

    .sum()

)


print(
    "\n--- CDC CHECK: SLD010H ---"
)


print(
    "2-11 hours:",
    f"{sld010h_2_to_11:,}"
)


print(
    "12 hours or more:",
    sld010h_12
)


print(
    "77 Refused:",
    sld010h_77
)


print(
    "99 Don't know:",
    sld010h_99
)


print(
    "Missing:",
    sld010h_missing
)


if sld010h_2_to_11 != 6408:

    raise ValueError(
        "SLD010H 2-11 count differs from CDC."
    )


if sld010h_12 != 46:

    raise ValueError(
        "SLD010H=12 count differs from CDC."
    )


if sld010h_77 != 0:

    raise ValueError(
        "SLD010H=77 Refused count differs from CDC."
    )


if sld010h_99 != 7:

    raise ValueError(
        "SLD010H=99 Don't know count differs from CDC."
    )


if sld010h_missing != 3:

    raise ValueError(
        "SLD010H missing count differs from CDC."
    )


print(
    "PASS: SLD010H matches CDC exactly."
)


# =================================================================================================
# 12. CDC FREQUENCY CHECK - SLQ050
# =================================================================================================

#
# Official CDC:
#
#   1 = Yes           1,582
#   2 = No            4,880
#   7 = Refused           0
#   9 = Don't know        2
#   Missing               0
#

SLQ050_EXPECTED = {

    1.0: 1582,

    2.0: 4880,

    9.0: 2

}


slq050_actual = (

    df["SLQ050"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


slq050_missing = int(

    df["SLQ050"]

    .isna()

    .sum()

)


print(
    "\n--- CDC CHECK: SLQ050 ---"
)


print(
    "Observed:",
    slq050_actual
)


print(
    "Missing:",
    slq050_missing
)


if slq050_actual != SLQ050_EXPECTED:

    raise ValueError(
        "SLQ050 distribution differs from CDC.\n\n"
        f"Expected:\n{SLQ050_EXPECTED}\n\n"
        f"Actual:\n{slq050_actual}"
    )


if slq050_missing != 0:

    raise ValueError(
        "SLQ050 missing count differs from CDC."
    )


print(
    "PASS: SLQ050 matches CDC exactly."
)


# =================================================================================================
# 13. CDC FREQUENCY CHECK - SLQ060
# =================================================================================================

#
# Official CDC:
#
#   1 = Yes             579
#   2 = No            5,872
#   7 = Refused           0
#   9 = Don't know       13
#   Missing               0
#

SLQ060_EXPECTED = {

    1.0: 579,

    2.0: 5872,

    9.0: 13

}


slq060_actual = (

    df["SLQ060"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


slq060_missing = int(

    df["SLQ060"]

    .isna()

    .sum()

)


print(
    "\n--- CDC CHECK: SLQ060 ---"
)


print(
    "Observed:",
    slq060_actual
)


print(
    "Missing:",
    slq060_missing
)


if slq060_actual != SLQ060_EXPECTED:

    raise ValueError(
        "SLQ060 distribution differs from CDC.\n\n"
        f"Expected:\n{SLQ060_EXPECTED}\n\n"
        f"Actual:\n{slq060_actual}"
    )


if slq060_missing != 0:

    raise ValueError(
        "SLQ060 missing count differs from CDC."
    )


print(
    "PASS: SLQ060 matches CDC exactly."
)


# =================================================================================================
# 14. VERIFY NO SOURCE VALUES REQUIRE CORRECTION
# =================================================================================================

#
# SLQ_H has:
#
#   0 tiny artifacts
#   0 true zeros
#
# Therefore there is NO value-correction step.
#
# Current dataframe must remain exactly numerically equivalent
# to the pandas-decoded source.
#

source_value_failures = []


for col in EXPECTED_COLUMNS:

    source_values = (

        source_df[col]

        .to_numpy(
            dtype=float
        )

    )


    current_values = (

        df[col]

        .to_numpy(
            dtype=float
        )

    )


    same = np.allclose(

        source_values,

        current_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        source_value_failures.append(
            col
        )


if source_value_failures:

    raise ValueError(
        "Unexpected source values changed:\n"
        + str(
            source_value_failures
        )
    )


print(
    "PASS: No source/scientific values changed."
)


# =================================================================================================
# 15. MISSING-POSITION FIDELITY
# =================================================================================================

missing_position_errors = 0


for col in EXPECTED_COLUMNS:

    source_missing = (

        source_df[col]

        .isna()

        .to_numpy()

    )


    current_missing = (

        df[col]

        .isna()

        .to_numpy()

    )


    missing_position_errors += int(

        (
            source_missing
            !=
            current_missing
        )

        .sum()

    )


if missing_position_errors != 0:

    raise ValueError(
        "Missing positions changed during cleaning."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# =================================================================================================
# 16. CONVERT PROVEN WHOLE-NUMBER FIELDS TO NULLABLE Int64
# =================================================================================================

#
# This is where:
#
#     SEQN = 73557.0
#
# becomes:
#
#     SEQN = 73557
#
#
# and:
#
#     SLQ050 = 1.0
#
# becomes:
#
#     SLQ050 = 1
#
#
# It is safe ONLY because we already proved all values
# are mathematically integral.
#

for col in INTEGER_COLUMNS:

    values = (

        df[col]

        .dropna()

        .to_numpy(
            dtype=float
        )

    )


    if len(values) > 0:

        integral_mask = np.isclose(

            values,

            np.round(
                values
            ),

            rtol=0,

            atol=1e-12

        )


        if not integral_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )


    df[col] = (

        df[col]

        .astype("Int64")

    )


print(
    "PASS: Proven whole-number/code fields converted to Int64."
)


# =================================================================================================
# 17. PRE-EXPORT NUMERIC FIDELITY
# =================================================================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    source_values = (

        source_df[col]

        .to_numpy(
            dtype=float
        )

    )


    current_values = (

        pd.to_numeric(
            df[col],
            errors="coerce"
        )

        .to_numpy(
            dtype=float,
            na_value=np.nan
        )

    )


    same = np.allclose(

        source_values,

        current_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Pre-export numeric fidelity failures:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export numeric fidelity."
)


# =================================================================================================
# 18. EXPORT CORRECTED CSV
# =================================================================================================

#
# CRITICAL GENERAL NHANES RULE:
#
# No:
#
#     float_format=
#
# No:
#
#     round(...)
#
# This protects genuine decimal variables in other components.
#

df.to_csv(

    CSV_FILE,

    index=False,

    na_rep=""

)


print(
    "\nCSV created:"
)


print(
    CSV_FILE
)


# =================================================================================================
# 19. RE-READ FINAL CSV WITH ROUND-TRIP NUMERIC PARSER
# =================================================================================================

roundtrip_df = pd.read_csv(

    CSV_FILE,

    low_memory=False,

    float_precision="round_trip"

)


if roundtrip_df.shape != (

    EXPECTED_ROWS,

    len(EXPECTED_COLUMNS)

):

    raise ValueError(
        "Final CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    !=
    EXPECTED_COLUMNS
):

    raise ValueError(
        "Final CSV column order changed."
    )


print(
    "PASS: Final CSV structure preserved."
)


# =================================================================================================
# 20. XPT -> CSV NUMERIC FIDELITY
# =================================================================================================

roundtrip_failures = []


for col in EXPECTED_COLUMNS:

    source_values = (

        source_df[col]

        .to_numpy(
            dtype=float
        )

    )


    final_values = (

        roundtrip_df[col]

        .to_numpy(
            dtype=float
        )

    )


    same = np.allclose(

        source_values,

        final_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        roundtrip_failures.append(
            col
        )


if roundtrip_failures:

    raise ValueError(
        "XPT -> CSV numeric fidelity failures:\n"
        + str(
            roundtrip_failures
        )
    )


print(
    "PASS: All four variables round-trip exactly."
)


# =================================================================================================
# 21. READ FINAL CSV AS RAW TEXT
# =================================================================================================

raw_csv = pd.read_csv(

    CSV_FILE,

    dtype=str,

    keep_default_na=False

)


# =================================================================================================
# 22. CHECK UNWANTED ".0"
# =================================================================================================

dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    bad_mask = (

        raw_csv[col]
        .ne("")

        &

        raw_csv[col]
        .str.endswith(".0")

    )


    bad_count = int(
        bad_mask.sum()
    )


    if bad_count > 0:

        dot_zero_errors[
            col
        ] = bad_count


if dot_zero_errors:

    raise ValueError(
        "Unnecessary '.0' formatting remains:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' formatting."
)


# =================================================================================================
# 23. FINAL SEQN TEXT FIDELITY
# =================================================================================================

if (
    raw_csv["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Final CSV contains blank SEQN."
    )


if (
    raw_csv["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN contains '.0'."
    )


if (

    ~raw_csv["SEQN"]
    .str.fullmatch(r"\d+")

).any():

    raise ValueError(
        "Final CSV contains invalid SEQN formatting."
    )


seqn_numeric = pd.to_numeric(

    raw_csv["SEQN"],

    errors="raise"

)


if seqn_numeric.min() != 73557:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if seqn_numeric.max() != 83729:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


print(
    "PASS: Final SEQN is clean."
)


# =================================================================================================
# 24. FINAL TINY-XPORT CHECK
# =================================================================================================

final_tiny_total = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(

        raw_csv[col]
        .where(
            raw_csv[col] != "",
            np.nan
        ),

        errors="coerce"

    )


    final_tiny_total += int(

        values
        .eq(TINY_VALUE)
        .sum()

    )


if final_tiny_total != 0:

    raise ValueError(
        "Tiny XPORT artifacts exist in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts in final CSV."
)


# =================================================================================================
# 25. FINAL TRUE-ZERO CHECK
# =================================================================================================

final_zero_total = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(

        raw_csv[col]
        .where(
            raw_csv[col] != "",
            np.nan
        ),

        errors="coerce"

    )


    final_zero_total += int(

        values
        .eq(0)
        .sum()

    )


if final_zero_total != 0:

    raise ValueError(
        "Unexpected numeric zeros exist in final CSV."
    )


print(
    "PASS: Final CSV contains no unexpected true zeros."
)


# =================================================================================================
# 26. SCIENTIFIC NOTATION CHECK
# =================================================================================================

scientific_notation_errors = {}


for col in EXPECTED_COLUMNS:

    count = int(

        raw_csv[col]

        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )

        .sum()

    )


    if count > 0:

        scientific_notation_errors[
            col
        ] = count


if scientific_notation_errors:

    raise ValueError(
        "Unexpected scientific notation found:\n"
        + str(
            scientific_notation_errors
        )
    )


print(
    "PASS: No scientific notation."
)


# =================================================================================================
# 27. FINAL MISSING-POSITION CHECK
# =================================================================================================

final_missing_position_errors = 0


for col in EXPECTED_COLUMNS:

    source_missing = (

        source_df[col]

        .isna()

        .to_numpy()

    )


    csv_missing = (

        raw_csv[col]

        .eq("")

        .to_numpy()

    )


    final_missing_position_errors += int(

        (
            source_missing
            !=
            csv_missing
        )

        .sum()

    )


if final_missing_position_errors != 0:

    raise ValueError(
        "Final CSV missing positions differ from source XPT."
    )


print(
    "PASS: Missing positions preserved through CSV export."
)


# =================================================================================================
# 28. FINAL CDC FREQUENCY RECHECK
# =================================================================================================

final_numeric = pd.read_csv(

    CSV_FILE,

    low_memory=False,

    float_precision="round_trip"

)


# -------------------------------------------------------------------------------------------------
# SLD010H
# -------------------------------------------------------------------------------------------------

if int(

    final_numeric["SLD010H"]

    .between(
        2,
        11
    )

    .sum()

) != 6408:

    raise ValueError(
        "Final SLD010H 2-11 count differs."
    )


if int(

    final_numeric["SLD010H"]

    .eq(12)

    .sum()

) != 46:

    raise ValueError(
        "Final SLD010H=12 count differs."
    )


if int(

    final_numeric["SLD010H"]

    .eq(99)

    .sum()

) != 7:

    raise ValueError(
        "Final SLD010H=99 count differs."
    )


if int(

    final_numeric["SLD010H"]

    .isna()

    .sum()

) != 3:

    raise ValueError(
        "Final SLD010H missing count differs."
    )


# -------------------------------------------------------------------------------------------------
# SLQ050
# -------------------------------------------------------------------------------------------------

final_slq050 = (

    final_numeric["SLQ050"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


if final_slq050 != {

    1: 1582,

    2: 4880,

    9: 2

}:

    raise ValueError(
        "Final SLQ050 distribution differs."
    )


if int(

    final_numeric["SLQ050"]

    .isna()

    .sum()

) != 0:

    raise ValueError(
        "Final SLQ050 missing count differs."
    )


# -------------------------------------------------------------------------------------------------
# SLQ060
# -------------------------------------------------------------------------------------------------

final_slq060 = (

    final_numeric["SLQ060"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


if final_slq060 != {

    1: 579,

    2: 5872,

    9: 13

}:

    raise ValueError(
        "Final SLQ060 distribution differs."
    )


if int(

    final_numeric["SLQ060"]

    .isna()

    .sum()

) != 0:

    raise ValueError(
        "Final SLQ060 missing count differs."
    )


print(
    "PASS: Final CSV frequencies match CDC."
)


# =================================================================================================
# 29. FINAL SUMMARY
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "NHANES 2013-2014 SLQ_H CONVERSION COMPLETE"
)


print(
    "=" * 110
)


print(
    "Cycle:"
)


print(
    "  2013-2014"
)


print(
    ""
)


print(
    "Component:"
)


print(
    "  Questionnaire - Sleep Disorders"
)


print(
    ""
)


print(
    "Eligible population:"
)


print(
    "  Age 16 years and older"
)


print(
    ""
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    ""
)


print(
    "Fields:"
)


print(
    "  SEQN"
)


print(
    "  SLD010H"
)


print(
    "  SLQ050"
)


print(
    "  SLQ060"
)


print(
    ""
)


print(
    "Original tiny XPORT artifacts:",
    tiny_total
)


print(
    "Validated zero restorations:",
    0
)


print(
    "True zero values:",
    0
)


print(
    ""
)


print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    ""
)


print(
    "SEQN '.0': 0"
)


print(
    "Duplicate SEQN: 0"
)


print(
    "Unexpected source-value changes: 0"
)


print(
    "Missing-position changes: 0"
)


print(
    "Scientific notation: 0"
)


print(
    "Round-trip numeric failures: 0"
)


print(
    ""
)


print(
    "CDC validation:"
)


print(
    "  SLD010H: PASS"
)


print(
    "  SLQ050:  PASS"
)


print(
    "  SLQ060:  PASS"
)


print(
    "=" * 110
)

NHANES 2013-2014 SLQ_H XPT -> CSV
Rows: 6,464
Columns: 4
Column names: ['SEQN', 'SLD010H', 'SLQ050', 'SLQ060']
PASS: Structure = 6,464 rows x 4 columns.
PASS: All four fields loaded numerically.

--- XPORT TINY-ZERO CHECK ---
SEQN       0
SLD010H    0
SLQ050     0
SLQ060     0

Total tiny XPORT values: 0
PASS: No SAS XPORT tiny-zero artifacts.

--- TRUE NUMERIC ZERO CHECK ---
SEQN       0
SLD010H    0
SLQ050     0
SLQ060     0

Total true numeric zeros: 0
PASS: No zero-coded observations exist in SLQ_H.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional values
SLD010H: 0 fractional values
SLQ050: 0 fractional values
SLQ060: 0 fractional values
PASS: No genuine decimal variables exist in SLQ_H.

--- FIELD TYPE PLAN ---
Whole-number/code fields: ['SEQN', 'SLD010H', 'SLQ050', 'SLQ060']
Genuine decimal fields: []

--- SEQN SOURCE CHECK ---
Minimum: 73557
Maximum: 83729
Missing: 0
Duplicates: 0
PASS: SEQN 73557-83729 validated.

--- CDC CHECK: SLD010H ---
2-11 hours: 6,408
12 hours or more: